# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring the FAIR^2 mass spectrometry dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Print dataset title and description
print(f"{dataset.metadata.name}: {dataset.metadata.description}\n")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List all available record sets and their @id
print("Record sets in the dataset:")
for record_set in dataset.metadata.record_sets:
    print(f"- Name: {record_set.name}, @id: {record_set.id}")

# Optionally, list fields for each record set
print("\nFields and columns for each record set:")
for record_set in dataset.metadata.record_sets:
    print(f"\nRecord set '{record_set.name}' (@id={record_set.id}):")
    for field in record_set.fields:
        cols = ', '.join(col.id for col in getattr(field, 'columns', []) if hasattr(field, 'columns'))
        print(f"  - Field: {field.name}, @id: {field.id}, Columns: [{cols}]")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Get all record set @id's
record_set_ids = [record_set.id for record_set in dataset.metadata.record_sets]

# For this notebook, select the main protein abundance record set. Adjust selection as appropriate.
main_record_set_id = None
for rs in dataset.metadata.record_sets:
    # Choose a record set whose ID or name indicates it contains tabular protein abundance data
    # Typical candidates contain 'protein', 'abundance', or 'ms' in the @id or name, adapt as needed.
    if ('protein' in rs.id.lower()) or ('abundance' in rs.id.lower()) or ('tabular' in rs.id.lower()):
        main_record_set_id = rs.id
        break
if main_record_set_id is None and len(record_set_ids) > 0:
    main_record_set_id = record_set_ids[0]

print(f"Using record set: {main_record_set_id}")

# Load the records for each record set into DataFrames
dataframes = {}
for rsid in record_set_ids:
    records = list(dataset.records(record_set=rsid))
    if records:
        dataframes[rsid] = pd.DataFrame(records)
    else:
        dataframes[rsid] = pd.DataFrame()

print(f"\nColumns available in record set {main_record_set_id}:")
print(dataframes[main_record_set_id].columns.tolist())
print("\nSample rows:")
dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes removing outliers, transforming data, or grouping data by key attributes to prepare for further analysis.

In [ ]:
# Select a numeric (quantitative) field for demonstration
df = dataframes[main_record_set_id]

# Try to auto-select a numeric field by typical column names or dtype
numeric_field = None
for col in df.columns:
    if ('abundance' in col.lower()) or ('count' in col.lower()) or ('mw' in col.lower()) or ('number' in col.lower()):
        # Check if the column is convertible to numeric
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field = col
            break
        # Try to convert if not
        try:
            df[col] = pd.to_numeric(df[col], errors='coerce')
            if pd.api.types.is_numeric_dtype(df[col]):
                numeric_field = col
                break
        except Exception:
            continue
if numeric_field is None and len(df.columns) > 0:
    # Pick first column as fallback
    numeric_field = df.columns[0]
    df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')

# Filtering: keep only records with numeric_field > some threshold
threshold = df[numeric_field].quantile(0.75) if df[numeric_field].notnull().sum() > 20 else 10
filtered_df = df[df[numeric_field] > threshold]

print(f"Filtered records with '{numeric_field}' > {threshold:.2f} (top quartile):")
print(filtered_df.head())

# Normalization (z-score)
filtered_df[f"{numeric_field}_normalized"] = (
    filtered_df[numeric_field] - filtered_df[numeric_field].mean()
) / filtered_df[numeric_field].std()

print(f"\nNormalized '{numeric_field}' for filtered records:")
print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Optionally group by a categorical field, such as 'description' or 'accession' if present
group_field = None
for possible in ["description", "accession", "sample", "category"]:
    if possible in df.columns:
        group_field = possible
        break

if group_field:
    grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
    print(f"\nGrouped data by '{group_field}':")
    print(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of the chosen numeric field
plt.figure(figsize=(8, 4))
sns.histplot(df[numeric_field].dropna(), bins=30, kde=True, color='skyblue')
plt.title(f"Distribution of '{numeric_field}' in record set {main_record_set_id}")
plt.xlabel(numeric_field)
plt.ylabel("Count")
plt.show()

# If possible, boxplot by a categorical variable
if group_field:
    plt.figure(figsize=(10, 6))
    # Show only top 10 most common categories
    top_groups = df[group_field].value_counts().index[:10]
    sns.boxplot(x=group_field, y=numeric_field, data=df[df[group_field].isin(top_groups)], palette='Set2')
    plt.title(f"{numeric_field} by {group_field}")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- The FAIR^2 dataset on mass spectrometry analysis of extracellular vesicles offers rich tabular data on protein abundance and related variables.
- We identified and loaded record sets by Croissant `@id`, and programmatically extracted both field lists and data.
- Example EDA demonstrated how to filter highly abundant proteins, normalize values, and visualize distributions using standard Python tools.
- This notebook can be easily extended to more advanced analyses such as clustering, feature engineering, and multi-record set integration—using the semantic structure from the Croissant schema and maintaining all references by `@id` for reproducibility and traceability.